In [7]:
!pip install pyspark

  Using cached pyspark-4.0.1.tar.gz (434.2 MB)
  Preparing metadata (setup.py) ... done
  Using cached py4j-0.10.9.9-py2.py3-none-any.whl (203 kB)
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.7
    Uninstalling py4j-0.10.9.7:
      Successfully uninstalled py4j-0.10.9.7
  DEPRECATION: pyspark is being installed using the legacy 'setup.py install' method, because it does not have a 'pyproject.toml' and the 'wheel' package is not installed. pip 23.1 will enforce this behaviour change. A possible replacement is to enable the '--use-pep517' option. Discussion can be found at https://github.com/pypa/pip/issues/8559
  Running setup.py install for pyspark ... done

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [1]:
!rm -rf output_report/

In [1]:
%env JAVA_HOME=/opt/homebrew/opt/openjdk@17

env: JAVA_HOME=/opt/homebrew/opt/openjdk@17


In [2]:
%env SPARK_HOME=/opt/homebrew/Cellar/apache-spark/4.0.1_1/libexec

env: SPARK_HOME=/opt/homebrew/Cellar/apache-spark/4.0.1_1/libexec


In [ ]:
%env SPARK_HOME=/Users/mgcrp/spark-3.4.2-bin-hadoop3

In [3]:
!echo $JAVA_HOME

/opt/homebrew/opt/openjdk@17


In [4]:
!echo $SPARK_HOME

/opt/homebrew/Cellar/apache-spark/4.0.1_1/libexec


In [5]:
from pyspark.sql import SparkSession

import os

from pyspark.sql import functions as f
from pyspark.sql.dataframe import DataFrame
from pyspark.sql.functions import collect_set, to_json, struct
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DoubleType

trips_schema = StructType([
    StructField('vendor_id', StringType(), True),
    StructField('tpep_pickup_datetime', TimestampType(), True),
    StructField('tpep_dropoff_datetime', TimestampType(), True),
    StructField('passenger_count', IntegerType(), True),
    StructField('trip_distance', DoubleType(), True),
    StructField('ratecode_id', IntegerType(), True),
    StructField('store_and_fwd_flag', StringType(), True),
    StructField('pulocation_id', IntegerType(), True),
    StructField('dolocation_id', IntegerType(), True),
    StructField('payment_type', IntegerType(), True),
    StructField('fare_amount', DoubleType(), True),
    StructField('extra', DoubleType(), True),
    StructField('mta_tax', DoubleType(), True),
    StructField('tip_amount', DoubleType(), True),
    StructField('tolls_amount', DoubleType(), True),
    StructField('improvement_surcharge', DoubleType(), True),
    StructField('total_amount', DoubleType(), True),
    StructField('congestion_surcharge', DoubleType()),
])


def foreach_batch_function(df: DataFrame, epoch_id): # функция для записи батчей
    df.write.mode("append").json("output_report") # датафрейм будет писаться в папку output_report методом append


def main(spark: SparkSession):
    jsonOptions = {"timestampFormat": "yyyy-MM-dd'T'HH:mm:ss.sss'Z'"} # указываем опции json-сериализации

    fields = list(map(lambda x: f"json_message.{x.name}", trips_schema.fields)) 

    df = spark \
        .readStream \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "localhost:9092") \
        .option("subscribe", "taxi") \
        .option("startingOffsets", "earliest") \
        .option("maxOffsetsPerTrigger", 1000) \
        .load() \
        .select(f.from_json(f.col("value").cast("string"), trips_schema, jsonOptions).alias("json_message")) \
        .select(fields)

    # пишем на диск
    writer = df \
        .writeStream \
        .foreachBatch(foreach_batch_function) \
        .trigger(processingTime='10 seconds') \
        .option("path", "output_report") \
        .outputMode("append") \
        .start()

    writer.awaitTermination()

In [6]:
main(
    SparkSession \
    .builder \
    .config("spark.driver.host", "localhost")\
    .appName("streaming_job") \
    # .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.2.1") \

    # .config("spark.jars.packages", "org.apache.kafka:kafka-clients:3.7.0") \
    # .config("spark.jars.packages", "org.apache.commons:commons-pool2:2.12.0") \
    # .config("spark.jars.packages", "org.apache.spark:spark-token-provider-kafka-0-10_2.12:3.3.0") \
    # .config("spark.jars.packages", "org.apache.spark:spark-streaming-kafka-0-10-assembly_2.12:3.3.0") \
        
    .config("spark.driver.bindAddress","localhost") \
    .config("spark.ui.port","4050") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/29 14:38:17 WARN Utils: Your hostname, MacBook-Pro-Popov.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.5 instead (on interface en0)
25/11/29 14:38:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/29 14:38:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


AnalysisException: Failed to find data source: kafka. Please deploy the application as per the deployment section of Structured Streaming + Kafka Integration Guide.

25/11/29 21:05:16 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 928337 ms exceeds timeout 120000 ms
25/11/29 21:05:16 WARN SparkContext: Killing executors is not supported by current scheduler.
25/11/29 21:05:22 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

In [29]:
from pyspark.sql import SparkSession

import os

from pyspark.sql import functions as f
from pyspark.sql.dataframe import DataFrame
from pyspark.sql.functions import collect_set, to_json, struct
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DoubleType

trips_schema = StructType([
    StructField('vendor_id', StringType(), True),
    StructField('tpep_pickup_datetime', TimestampType(), True),
    StructField('tpep_dropoff_datetime', TimestampType(), True),
    StructField('passenger_count', IntegerType(), True),
    StructField('trip_distance', DoubleType(), True),
    StructField('ratecode_id', IntegerType(), True),
    StructField('store_and_fwd_flag', StringType(), True),
    StructField('pulocation_id', IntegerType(), True),
    StructField('dolocation_id', IntegerType(), True),
    StructField('payment_type', IntegerType(), True),
    StructField('fare_amount', DoubleType(), True),
    StructField('extra', DoubleType(), True),
    StructField('mta_tax', DoubleType(), True),
    StructField('tip_amount', DoubleType(), True),
    StructField('tolls_amount', DoubleType(), True),
    StructField('improvement_surcharge', DoubleType(), True),
    StructField('total_amount', DoubleType(), True),
    StructField('congestion_surcharge', DoubleType()),
])


def foreach_batch_function(df: DataFrame, epoch_id): # функция для записи батчей
    df.write.mode("append").json("output_report") # датафрейм будет писаться в папку output_report методом append


def main(spark: SparkSession):
    jsonOptions = {"timestampFormat": "yyyy-MM-dd'T'HH:mm:ss.sss'Z'"} # указываем опции json-сериализации

    fields = list(map(lambda x: f"json_message.{x.name}", trips_schema.fields)) 

    df = spark \
        .readStream \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "localhost:9092") \
        .option("subscribe", "taxi") \
        .option("startingOffsets", "latest") \
        .option("maxOffsetsPerTrigger", 1000) \
        .load() \
        .select(f.from_json(f.col("value").cast("string"), trips_schema, jsonOptions).alias("json_message")) \
        .select(fields)
    
    mart = df \
        .groupBy('payment_type', f.to_date('tpep_pickup_datetime').alias('dt')) \
        .agg(f.count(f.col('*')).alias('cnt')) \
        .select(f.col('payment_type'), f.col('cnt'), f.col('dt')) \
        .orderBy(f.col('payment_type'), f.col('dt')) \
        .select(to_json(struct("payment_type", "cnt", "dt")).alias('value'))

    writer = mart \
        .writeStream \
        .trigger(processingTime='10 seconds') \
        .format("console") \
        .outputMode("complete") \
        .option("truncate", "false") \
        .start()

    writer.awaitTermination()

In [39]:
main(
    SparkSession \
    .builder \
    .config("spark.driver.host", "localhost")\
    .appName("streaming_job") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.2.0") \
    .config("spark.driver.bindAddress","localhost") \
    .config("spark.ui.port","4050") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()
)

Py4JJavaError: An error occurred while calling o55.load.
: java.lang.NoClassDefFoundError: Could not initialize class org.apache.spark.sql.kafka010.KafkaSourceProvider$
	at org.apache.spark.sql.kafka010.KafkaSourceProvider.org$apache$spark$sql$kafka010$KafkaSourceProvider$$validateStreamOptions(KafkaSourceProvider.scala:338)
	at org.apache.spark.sql.kafka010.KafkaSourceProvider.sourceSchema(KafkaSourceProvider.scala:71)
	at org.apache.spark.sql.execution.datasources.DataSource.sourceSchema(DataSource.scala:244)
	at org.apache.spark.sql.execution.datasources.DataSource.sourceInfo$lzycompute(DataSource.scala:129)
	at org.apache.spark.sql.execution.datasources.DataSource.sourceInfo(DataSource.scala:129)
	at org.apache.spark.sql.execution.streaming.StreamingRelation$.apply(StreamingRelation.scala:37)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfun$apply$1.applyOrElse(ResolveDataSource.scala:86)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfun$appl

In [13]:
from pyspark.sql import SparkSession

import os

from pyspark.sql import functions as f
from pyspark.sql.dataframe import DataFrame
from pyspark.sql.functions import collect_set, to_json, struct
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DoubleType

trips_schema = StructType([
    StructField('vendor_id', StringType(), True),
    StructField('tpep_pickup_datetime', TimestampType(), True),
    StructField('tpep_dropoff_datetime', TimestampType(), True),
    StructField('passenger_count', IntegerType(), True),
    StructField('trip_distance', DoubleType(), True),
    StructField('ratecode_id', IntegerType(), True),
    StructField('store_and_fwd_flag', StringType(), True),
    StructField('pulocation_id', IntegerType(), True),
    StructField('dolocation_id', IntegerType(), True),
    StructField('payment_type', IntegerType(), True),
    StructField('fare_amount', DoubleType(), True),
    StructField('extra', DoubleType(), True),
    StructField('mta_tax', DoubleType(), True),
    StructField('tip_amount', DoubleType(), True),
    StructField('tolls_amount', DoubleType(), True),
    StructField('improvement_surcharge', DoubleType(), True),
    StructField('total_amount', DoubleType(), True),
    StructField('congestion_surcharge', DoubleType()),
])


def foreach_batch_function(df: DataFrame, epoch_id): # функция для записи батчей
    df.write.mode("append").json("output_report") # датафрейм будет писаться в папку output_report методом append


def main(spark: SparkSession):
    jsonOptions = {"timestampFormat": "yyyy-MM-dd'T'HH:mm:ss.sss'Z'"} # указываем опции json-сериализации

    fields = list(map(lambda x: f"json_message.{x.name}", trips_schema.fields)) 

    df = spark \
        .readStream \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "localhost:9092") \
        .option("subscribe", "taxi") \
        .option("startingOffsets", "latest") \
        .option("maxOffsetsPerTrigger", 1000) \
        .load() \
        .select(f.from_json(f.col("value").cast("string"), trips_schema, jsonOptions).alias("json_message")) \
        .select(fields)
    
    mart = df \
        .groupBy('payment_type', f.to_date('tpep_pickup_datetime').alias('dt')) \
        .agg(f.count(f.col('*')).alias('cnt')) \
        .select(f.col('payment_type'), f.col('cnt'), f.col('dt')) \
        .orderBy(f.col('payment_type'), f.col('dt')) \
        .select(to_json(struct("payment_type", "cnt", "dt")).alias('value'))

    writer = mart \
        .writeStream \
        .trigger(processingTime='10 seconds') \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "localhost:29092") \
        .option("topic", "report") \
        .option("checkpointLocation", "streaming-job-checkpoint") \
        .outputMode("complete") \
        .start()

    writer.awaitTermination()

In [15]:
main(
    SparkSession \
    .builder \
    .config("spark.driver.host", "localhost")\
    .appName("streaming_job") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.2.0") \
    .config("spark.driver.bindAddress","localhost") \
    .config("spark.ui.port","4050") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()
)

24/12/03 19:57:25 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
24/12/03 19:57:25 WARN AdminClientConfig: The configuration 'key.deserializer' was supplied but isn't a known config.
24/12/03 19:57:25 WARN AdminClientConfig: The configuration 'value.deserializer' was supplied but isn't a known config.
24/12/03 19:57:25 WARN AdminClientConfig: The configuration 'enable.auto.commit' was supplied but isn't a known config.
24/12/03 19:57:25 WARN AdminClientConfig: The configuration 'max.poll.records' was supplied but isn't a known config.
24/12/03 19:57:25 WARN AdminClientConfig: The configuration 'auto.offset.reset' was supplied but isn't a known config.
24/12/03 19:57:25 ERROR MicroBatchExecution: Query [id = 8f8b0fed-2055-43e2-965f-02fd5a3bfe41, runId = 5ff69e81-abad-43d7-b6c7-11b71ba5c5bf] terminated with error
java.lang.NoSuchMethodError: org.apache.spark.sql.types.StructType.toAttributes()Lscala/collection

StreamingQueryException: [STREAM_FAILED] Query [id = 8f8b0fed-2055-43e2-965f-02fd5a3bfe41, runId = 5ff69e81-abad-43d7-b6c7-11b71ba5c5bf] terminated with exception: org.apache.spark.sql.types.StructType.toAttributes()Lscala/collection/Seq;

In [17]:
spark.stop()

-------------------------------------------
Batch: 19
-------------------------------------------
+-----------------------------------------------+
|value                                          |
+-----------------------------------------------+
|{"payment_type":1,"cnt":9,"dt":"2020-02-03"}   |
|{"payment_type":1,"cnt":5,"dt":"2020-03-08"}   |
|{"payment_type":1,"cnt":823,"dt":"2020-04-01"} |
|{"payment_type":1,"cnt":1024,"dt":"2020-04-02"}|
|{"payment_type":2,"cnt":5,"dt":"2020-02-03"}   |
|{"payment_type":2,"cnt":5,"dt":"2020-03-08"}   |
|{"payment_type":2,"cnt":554,"dt":"2020-04-01"} |
|{"payment_type":2,"cnt":649,"dt":"2020-04-02"} |
|{"payment_type":3,"cnt":14,"dt":"2020-04-01"}  |
|{"payment_type":3,"cnt":16,"dt":"2020-04-02"}  |
|{"payment_type":4,"cnt":4,"dt":"2020-04-01"}   |
|{"payment_type":4,"cnt":12,"dt":"2020-04-02"}  |
+-----------------------------------------------+



NameError: name 'spark' is not defined